# Chapitre 4 - Méthodes linéaires de classification
## Cours 2 : LDA de rang réduit et variables canoniques de Fisher

Ce notebook traite la section 4.3.3 du livre. La LDA ne fournit pas seulement une règle
de classification : elle fournit aussi des **projections de faible dimension** des
données, les plus informatives pour distinguer les classes. Nous reproduisons sur les
données de voyelles les figures 4.4, 4.8, 4.10 et 4.11, et illustrons la figure 4.9.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch04_classification as c4
set_style()

In [2]:
Xtr, ytr = datasets.load_vowel("train")
Xte, yte = datasets.load_vowel("test")
K, p = len(np.unique(ytr)), Xtr.shape[1]
print("K =", K, " p =", p, " N_train =", len(ytr), " N_test =", len(yte))

K = 11  p = 10  N_train = 528  N_test = 462


Pour représenter 11 classes, nous utilisons une palette de 11 couleurs : les 8 couleurs
catégorielles du projet, complétées par trois teintes neutres.

In [3]:
VOW = COLORS + ["#52514e", "#8c6d31", "#17becf"]
classes = np.arange(1, 12)

## 1. Pourquoi une réduction de dimension est possible

Rappelons (cours 1) que, après **blanchiment** $x^* = \mathbf{W}^{-1/2}x$ par la covariance
intra-classe $\mathbf{W} = \hat\Sigma$, la LDA classe au **centroïde le plus proche**
(corrigé par $\log\pi_k$).

Les $K$ centroïdes $\mu_1^*, \dots, \mu_K^*$ engendrent un sous-espace affine
$H_{K-1}$ de dimension au plus $K-1$. Décomposons $x^* - \mu_k^*$ en sa composante dans
la direction de $H_{K-1}$ et sa composante orthogonale. Par le théorème de Pythagore,
$$\|x^*-\mu_k^*\|^2 = \|P(x^*) - \mu_k^*\|^2 + \|x^* - P(x^*)\|^2,$$
où $P$ est la projection orthogonale sur $H_{K-1}$. Le second terme **ne dépend pas de
$k$** : il ne joue aucun rôle dans la comparaison.

> **À retenir.** La LDA peut s'effectuer intégralement dans $H_{K-1}$, de dimension
> $K - 1$. Avec $K = 3$ classes, on peut donc voir *toute* l'information utile de la LDA
> dans un plan, quelle que soit la dimension $p$ des données.

Quand $K - 1$ est encore grand (ici $K - 1 = 10 = p$), on cherche un sous-espace
$H_L \subset H_{K-1}$ de dimension $L$ optimal : celui où les centroïdes projetés sont
**le plus dispersés possible**, au sens de la variance.

## 2. Construction des variables canoniques

La procédure du livre comporte trois étapes :

1. calculer la matrice $\mathbf{M}$ ($K\times p$) des centroïdes de classe et la covariance
   intra-classe $\mathbf{W}$ ($p \times p$) ;
2. calculer $\mathbf{M}^* = \mathbf{M}\mathbf{W}^{-1/2}$ (centroïdes blanchis) ;
3. calculer $\mathbf{B}^*$, matrice de covariance (pondérée par les $\pi_k$) des lignes de
   $\mathbf{M}^*$, et sa décomposition spectrale
   $\mathbf{B}^* = \mathbf{V}^*\mathbf{D}_B\mathbf{V}^{*T}$ (valeurs propres décroissantes).

La $\ell$-ième **variable canonique** (ou coordonnée discriminante) est
$Z_\ell = v_\ell^T X$ avec $v_\ell = \mathbf{W}^{-1/2}v_\ell^*$.

Rappel : pour une matrice symétrique définie positive $\mathbf{W} = \mathbf{U}\mathbf{D}\mathbf{U}^T$,
on définit $\mathbf{W}^{-1/2} = \mathbf{U}\mathbf{D}^{-1/2}\mathbf{U}^T$ ; c'est l'unique
matrice symétrique définie positive telle que $(\mathbf{W}^{-1/2})^2 = \mathbf{W}^{-1}$.